# Email Triage RL — GRPO Training

**OpenEnv Hackathon 2026 — Team Ctrl-Alt-Defeat**

Trains `Qwen/Qwen3.5-2B` to triage emails using GRPO via TRL + Unsloth.  
Three specialist agents (Analyst → Router → Responder) are trained jointly with role-conditioned prompts.

Runtime: T4 ~40 min · A100 ~10 min

## 1. Setup

In [ ]:
%%capture
!pip install --upgrade pip --quiet
!pip install "transformers>=4.48.0" --force-reinstall --quiet
!pip install "trl>=0.15.0" "peft>=0.14.0" "accelerate>=0.34.0" bitsandbytes --quiet
!pip install unsloth --quiet
!pip install sentencepiece protobuf datasets wandb "openenv-core>=0.2.0" --quiet

In [ ]:
# After install, restart runtime once: Runtime → Restart, then Run all
import trl, peft, transformers
print(f"transformers={transformers.__version__}  trl={trl.__version__}  peft={peft.__version__}")

In [ ]:
import os, sys, torch

REPO = '/content/email-triage-env'
if not os.path.isdir(REPO):
    !git clone https://huggingface.co/spaces/Hk4crprasad/email-triage-env $REPO
else:
    !cd $REPO && git pull --quiet

sys.path.insert(0, REPO)
os.chdir(REPO)

assert torch.cuda.is_available(), "Switch to T4/A100: Runtime → Change runtime type"
p = torch.cuda.get_device_properties(0)
print(f"{p.name}  {p.total_memory/1e9:.1f} GB  CUDA {torch.version.cuda}")

## 2. Load Qwen3.5-2B

In [ ]:
MODEL_NAME = 'Qwen/Qwen3.5-2B'
LORA_RANK  = 16

# Pre-load AutoTokenizer — used for decode() since Qwen3_5Processor can't decode plain ids
from transformers import AutoTokenizer as _AutoTok
_text_tokenizer = _AutoTok.from_pretrained(MODEL_NAME, trust_remote_code=True)
if _text_tokenizer.pad_token is None:
    _text_tokenizer.pad_token = _text_tokenizer.eos_token

UNSLOTH_OK = False
try:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME, max_seq_length=2048,
        load_in_4bit=True, fast_inference=False,
    )
    model = FastLanguageModel.get_peft_model(
        model, r=LORA_RANK,
        target_modules=['q_proj','k_proj','v_proj','o_proj'],
        lora_alpha=LORA_RANK*2, lora_dropout=0.05, bias='none',
        use_gradient_checkpointing='unsloth',
    )
    UNSLOTH_OK = True
    print(f'Loaded via Unsloth  VRAM={torch.cuda.memory_allocated(0)/1e9:.2f} GB')
except Exception as e:
    print(f'Unsloth unavailable ({e}) — using HF transformers')
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                              bnb_4bit_compute_dtype=torch.float16)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb, device_map='auto', trust_remote_code=True)
    model = get_peft_model(base, LoraConfig(
        r=LORA_RANK, lora_alpha=LORA_RANK*2, lora_dropout=0.05,
        target_modules=['q_proj','k_proj','v_proj','o_proj'],
        bias='none', task_type='CAUSAL_LM'))
    print(f'Loaded via HF  VRAM={torch.cuda.memory_allocated(0)/1e9:.2f} GB')

## 3. Build training dataset

In [ ]:
import importlib
if 'train' in importlib.import_module('sys').modules:
    importlib.reload(importlib.import_module('train'))

from datasets import Dataset
from train import build_dataset, SYSTEM_PROMPT, format_email_prompt, REWARD_FUNCTIONS, _parse_action, evaluate_model

raw = build_dataset(['easy', 'medium', 'hard'], seed=42)
ds  = Dataset.from_list(raw)
by_t = {t: ds.filter(lambda x, t=t: x['task_id'] == t) for t in ['easy','medium','hard']}

print(f'{len(ds)} prompts — easy:{len(by_t["easy"])} medium:{len(by_t["medium"])} hard:{len(by_t["hard"])}')
print('Sample:', ds[0]['prompt'][1]['content'][:300])

In [ ]:
# Sanity check reward functions on a perfect action
perfect = '{"email_id":"%s","category":"%s","priority":%d,"department":"%s","escalate":false}' % (
    raw[0]['email_id'], raw[0]['gt_category'], raw[0]['gt_priority'], raw[0]['gt_department'])
kw = {k: [raw[0][k]] for k in raw[0]}
for fn in REWARD_FUNCTIONS:
    print(f'{fn.__name__:<25} {fn([perfect], **kw)[0]:+.2f}')

## 4. Baseline evaluation

In [ ]:
from server.environment import EmailTriageEnvironment
import json, time

EVAL_SEED = 99

def _tokenize(tokenizer, msgs, device):
    # Qwen3.5-2B: use apply_chat_template for the template, then _text_tokenizer for encoding.
    # Calling tokenizer(text) goes through Qwen3_5Processor which tries to load_image.
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return _text_tokenizer(text, return_tensors='pt', add_special_tokens=False)['input_ids'].to(device)

def run_episode(model, tokenizer, task_id, seed=EVAL_SEED):
    env = EmailTriageEnvironment()
    obs = env.reset(seed=seed, task_id=task_id)
    od  = obs.model_dump()
    while not od.get('done') and od.get('emails'):
        email = od['emails'][0]
        msgs = [
            {'role':'system','content':SYSTEM_PROMPT},
            {'role':'user','content':format_email_prompt(email, od.get('task_description',''))},
        ]
        ids = _tokenize(tokenizer, msgs, model.device)
        with torch.no_grad():
            out = model.generate(ids, max_new_tokens=200, do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id or tokenizer.pad_token_id)
        text = _text_tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)
        action = _parse_action(text) or {'email_id':email['email_id'],'category':'general','priority':3,'department':'support'}
        obs = env.step(action)
        od  = obs.model_dump()
    g = od.get('metadata',{}).get('grading',{})
    return g.get('final_score',0.0), g.get('dimension_scores',{})

if UNSLOTH_OK:
    model.disable_adapter_layers()
    FastLanguageModel.for_inference(model)

baseline = {}
for tid in ['easy','medium','hard']:
    score, dims = run_episode(model, tokenizer, tid)
    baseline[tid] = {'score': score, 'dims': dims}
    print(f'baseline {tid:<6} → {score:.4f}')

if UNSLOTH_OK:
    model.enable_adapter_layers()
    FastLanguageModel.for_training(model)

## 5. GRPO Training — curriculum easy → medium → hard

In [ ]:
import os
os.environ.setdefault('WANDB_PROJECT', 'email-triage-rl')
WANDB_KEY = ''  # paste your W&B API key here, or leave blank
if WANDB_KEY:
    os.environ['WANDB_API_KEY'] = WANDB_KEY
    REPORT = 'wandb'
else:
    REPORT = 'none'

In [ ]:
from trl import GRPOConfig, GRPOTrainer
import gc

STAGES = [('easy',200), ('medium',100), ('hard',100)]
trainers = []

for stage_idx, (tid, steps) in enumerate(STAGES):
    print(f'\n── Stage {stage_idx+1}/{len(STAGES)}: {tid} ({steps} steps) ──')
    print(f'   VRAM before: {torch.cuda.memory_allocated(0)/1e9:.2f} GB')

    cfg = GRPOConfig(
        output_dir                  = f'/content/checkpoints/{tid}',
        max_steps                   = steps,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        learning_rate               = 3e-6,
        lr_scheduler_type           = 'cosine',
        num_generations             = 4,
        max_new_tokens              = 200,
        temperature                 = 1.0,
        top_p                       = 0.95,
        max_grad_norm               = 0.1,
        warmup_steps                = 20,
        logging_steps               = 5,
        save_steps                  = 100,
        save_total_limit            = 1,
        seed                        = 42,
        use_vllm                    = False,
        report_to                   = REPORT,
    )
    trainer = GRPOTrainer(
        model=model, reward_funcs=REWARD_FUNCTIONS,
        args=cfg, train_dataset=by_t[tid], processing_class=tokenizer,
    )
    trainer.train()
    trainers.append((tid, trainer))

    if UNSLOTH_OK: FastLanguageModel.for_inference(model)
    s, _ = run_episode(model, tokenizer, tid)
    if UNSLOTH_OK: FastLanguageModel.for_training(model)
    gc.collect(); torch.cuda.empty_cache()
    print(f'   eval score: {s:.4f}  VRAM after: {torch.cuda.memory_allocated(0)/1e9:.2f} GB')

## 6. Final evaluation

In [ ]:
if UNSLOTH_OK: FastLanguageModel.for_inference(model)

trained = {}
for tid in ['easy','medium','hard']:
    score, dims = run_episode(model, tokenizer, tid)
    trained[tid] = {'score': score, 'dims': dims}
    print(f'trained {tid:<6} → {score:.4f}  {dims}')

print('\n── improvement ──')
for tid in ['easy','medium','hard']:
    b, t = baseline[tid]['score'], trained[tid]['score']
    print(f'{tid:<8} {b:.4f} → {t:.4f}  ({t-b:+.4f})')

## 7. Plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
os.makedirs('/content/plots', exist_ok=True)

# Training curve
fig, ax = plt.subplots(figsize=(10,4))
colors = {'easy':'#4caf50','medium':'#ff9800','hard':'#f44336'}
step_offset = 0
for tid, trainer in trainers:
    hist  = [h for h in trainer.state.log_history if 'reward' in h]
    steps = [h['step'] + step_offset for h in hist]
    rews  = [h['reward'] for h in hist]
    ax.plot(steps, rews, color=colors[tid], label=tid)
    if steps: step_offset = steps[-1]
ax.axhline(0.6, ls='--', color='gray', alpha=0.5, label='easy→medium')
ax.axhline(0.5, ls=':',  color='gray', alpha=0.5, label='medium→hard')
ax.set_xlabel('step'); ax.set_ylabel('reward')
ax.set_title('GRPO training reward — Qwen3.5-2B')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/plots/training_curve.png', dpi=150)
plt.show()

In [ ]:
# Score comparison
tasks = ['easy','medium','hard']
x, w = np.arange(3), 0.35
fig, ax = plt.subplots(figsize=(7,4))
b_scores = [baseline[t]['score'] for t in tasks]
t_scores = [trained[t]['score']  for t in tasks]
ax.bar(x - w/2, b_scores, w, label='Baseline (0-shot)', color='#90a4ae')
ax.bar(x + w/2, t_scores, w, label='After GRPO',        color='#1e88e5')
for i, (b,t) in enumerate(zip(b_scores, t_scores)):
    ax.text(i+w/2, t+0.01, f'+{t-b:.2f}', ha='center', fontsize=9, color='green', fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(tasks)
ax.set_ylim(0, 1.1); ax.set_ylabel('final score')
ax.set_title('Baseline vs GRPO — Qwen3.5-2B')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/plots/score_comparison.png', dpi=150)
plt.show()

In [ ]:
# Dimension breakdown (medium)
dims = sorted(set(baseline['medium']['dims']) | set(trained['medium']['dims']))
x = np.arange(len(dims))
fig, ax = plt.subplots(figsize=(9,4))
ax.bar(x-0.2, [baseline['medium']['dims'].get(d,0) for d in dims], 0.4, label='Baseline', color='#90a4ae')
ax.bar(x+0.2, [trained['medium']['dims'].get(d,0)  for d in dims], 0.4, label='Trained',  color='#1e88e5')
ax.set_xticks(x); ax.set_xticklabels(dims, rotation=20)
ax.set_ylabel('score'); ax.set_title('Per-dimension — medium task')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/plots/dimension_breakdown.png', dpi=150)
plt.show()

## 8. Save adapter

In [ ]:
ADAPTER_DIR = '/content/email-triage-grpo-adapter'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'Saved to {ADAPTER_DIR}')
!du -sh $ADAPTER_DIR

In [ ]:
import os
HF_TOKEN = os.environ.get('HF_TOKEN', '')
REPO_ID  = 'Hk4crprasad/email-triage-grpo'

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    model.push_to_hub(REPO_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(REPO_ID, token=HF_TOKEN)
    print(f'Pushed → https://huggingface.co/{REPO_ID}')
else:
    print('Set HF_TOKEN to push to Hub')

if REPORT == 'wandb':
    import wandb; wandb.finish()

print('\n── final scores ──')
for tid in ['easy','medium','hard']:
    b, t = baseline[tid]['score'], trained[tid]['score']
    print(f'{tid:<8} {b:.4f} → {t:.4f}  ({t-b:+.4f})')

## 9. Multi-Agent Training

Three specialist agents trained jointly with role-conditioned GRPO:
- **Analyst** — classify + extract signals  
- **Router** — route using analyst output (theory-of-mind)  
- **Responder** — draft using both agents' context (coalition)

In [ ]:
from train import build_multi_agent_dataset, MULTI_AGENT_REWARD_FUNCTIONS, train_multi_agent

ma_raw = build_multi_agent_dataset(['easy','medium','hard'], seed=42)
roles  = [e['role'] for e in ma_raw]
print(f'{len(ma_raw)} examples — analyst:{roles.count("analyst")} router:{roles.count("router")} responder:{roles.count("responder")}')

In [ ]:
import argparse

if UNSLOTH_OK: FastLanguageModel.for_training(model)

ma_args = argparse.Namespace(max_steps=200, seed=42, wandb=(REPORT=='wandb'))
ma_metrics = train_multi_agent(
    model=model, tokenizer=tokenizer,
    task_ids=['easy','medium','hard'],
    output_dir='/content/checkpoints/multi-agent',
    args=ma_args,
)
print(f'done: loss={ma_metrics["train_loss"]:.4f}  steps={ma_metrics["steps"]}')

In [ ]:
# Quick multi-agent eval on the phishing email (hard, seed=42, idx=2)
import json as _json
from server.multi_agent_env import (
    ANALYST_SYSTEM_PROMPT, ROUTER_SYSTEM_PROMPT, RESPONDER_SYSTEM_PROMPT,
    build_analyst_prompt, build_router_prompt, build_responder_prompt,
)
from server.reward import reward_coordination, reward_theory_of_mind, reward_coalition
from server.email_generator import generate_emails

if UNSLOTH_OK: FastLanguageModel.for_inference(model)

def run_agent(sys_p, usr_p):
    ids = _tokenize(tokenizer, [{'role':'system','content':sys_p},{'role':'user','content':usr_p}], model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=150, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id or tokenizer.pad_token_id)
    raw = _text_tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True).strip()
    try: return _json.loads(raw)
    except: pass
    for i,c in enumerate(raw):
        if c == '{':
            for j in range(len(raw)-1,i,-1):
                if raw[j]=='}':
                    try: return _json.loads(raw[i:j+1])
                    except: break
    return {}

emails_h, gts_h = generate_emails('hard', 42)
email   = emails_h[2]
gt      = {g.email_id: g for g in gts_h}[email.email_id]
email_d = email.model_dump()
td = 'Triage the email: classify, route, escalate if needed, draft response for urgent.'

a_out = run_agent(ANALYST_SYSTEM_PROMPT,   build_analyst_prompt(email_d, td))
r_out = run_agent(ROUTER_SYSTEM_PROMPT,    build_router_prompt(email_d, td, a_out))
resp  = run_agent(RESPONDER_SYSTEM_PROMPT, build_responder_prompt(email_d, td, a_out, r_out))

print(f'Email: {email.subject[:60]}')
print(f'Ground truth: category={gt.category}  dept={gt.department}')
print(f'\nAnalyst:   category={a_out.get("category")}  priority={a_out.get("priority")}  signals={a_out.get("signals",[])}')
print(f'Router:    dept={r_out.get("department")}  escalate={r_out.get("escalate")}  reason={r_out.get("routing_reason","")[:60]}')
print(f'Responder: tone={resp.get("tone")}  draft={str(resp.get("response_draft",""))[:80]}')

rc = reward_coordination(a_out.get('category',''), r_out.get('department',''), bool(r_out.get('escalate')))
rt = reward_theory_of_mind(a_out.get('signals',[]), r_out.get('routing_reason',''), a_out.get('category',''))
print(f'\nCoordination: {rc:+.2f}  Theory-of-Mind: {rt:+.2f}')